
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Advanced Data Quality Checks and Expectations in Spark Declarative Pipelines

## Overview

This demonstration showcases enterprise-grade data quality management using Spark Declarative Pipelines (Lakeflow) with advanced quality expectations and quarantine patterns. You will build a comprehensive medallion architecture that implements three critical data engineering patterns: range-based validation, automatic schema evolution, and zero-loss quarantine processing.

The demo simulates a real-world scenario where order data arrives with varying quality levels and evolving schemas. Using official Databricks patterns, you will process this data through bronze, silver, and gold layers while implementing robust quality controls that catch data anomalies, handle schema changes seamlessly, and ensure zero data loss through intelligent quarantine mechanisms.

This demonstration uses a two-run approach: Run 1 establishes a clean baseline with 157 perfect records to validate the quality framework, while Run 2 introduces 60 records containing intentional quality issues and schema evolution to demonstrate the system's resilience and monitoring capabilities.

## Learning Objectives

By the end of this demonstration, you will be able to:

- **Implement comprehensive data validation** using 6 quality expectations covering NOT NULL checks, numeric ranges, and date validations with proper constraint syntax
- **Handle automatic schema evolution** without pipeline code changes using flexible bronze layer design and schema hints
- **Build quarantine patterns** using official Databricks inverse logic to capture invalid records while maintaining zero data loss and detailed failure tracking
- **Monitor data quality metrics** and analyze expectation performance to identify specific data quality issues and develop remediation strategies

## Required - Select a Compute Environment

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 4**  
  - [How to select an environment version](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:** This notebook was **developed and tested using Serverless V4**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## Demo Architecture Overview

This demo demonstrates how to build a robust data pipeline using Databricks medallion architecture, focusing on data quality enforcement and schema flexibility.

### Key Patterns Demonstrated

**1. Advanced Data Quality Checks**
- Enforce comprehensive data quality rules: NOT NULL for business keys, numeric range validations, and temporal checks
- Invalid rows are systematically quarantined with detailed failure analysis for remediation workflows

**2. Seamless Schema Evolution**
- Handles new columns automatically—no pipeline code changes required
- Bronze layer stores all columns as STRING for maximum flexibility
- Silver layer performs safe type conversion with validation

**3. Zero-Loss Quarantine Pattern**
- Invalid records are captured in a dedicated quarantine table ensuring no data loss
- Tracks which specific rules failed for each record with detailed failure reasons
- Enables parallel processing of valid and invalid data streams

### Demo Pipeline Architecture

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/dq_pipeline_overview.png" alt="Complete Pipeline Run 1" width="1200">

### Demo Execution Flow

- **Run 1:** Clean baseline data (157 records) — all pass quality checks, establishes 100% quality baseline
- **Run 2:** Data with quality issues (60 records, 2 new columns) — demonstrates issue detection and schema evolution

## A. Setup

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size:1.1em;">
    Option 1 - Databricks Academy Provided Workspace (Vocareum Workspace)
  </strong>
  <details>
  <div style="color:#333;">

If you are running this notebook in a <strong>Databricks Academy provided Vocareum workspace</strong>, your Unity Catalog catalog is already created for you.

Your catalog name matches your Vocareum username and looks like: <strong>labuser12345</strong> (series of unique numbers)
  </div>
  </details>
</div>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size:1.1em;">
    Option 2 - Other Workspaces or Databricks Free Edition
  </strong>
  <details>
  <div style="color:#333;">

If you are running this notebook in your own Databricks workspace or Databricks Free Edition, the setup will
<strong>create a Unity Catalog catalog and schema for you</strong>. **Create catalog permission is required.**

The catalog name is derived from your Databricks username and follows this pattern: <strong>labuser_username</strong>
  </div>
  </details>
</div>

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Do Not Run in Production Environments</strong>
  <div style="color:#333;">
  <ul>
      <li>Only run this notebook in <strong>development or sandbox workspaces</strong>.</li>
      <li>Do not run this in production environments. The setup script creates a catalog and schemas in your workspace.</li>
  </ul>
  </div>
</div>

### A1. Configure Your Catalog and Schema

Run the cell below to initialize your environment. This setup step performs the following actions:

- **Creates or validates catalog access** when running outside of a Databricks provided Vocareum workspace
- **Creates four schemas** in your specified catalog:
    - **dq_1_bronze** - Raw data ingestion layer
    - **dq_2_silver** - Validated and transformed data layer  
    - **dq_3_gold** - Analytics-ready aggregated data layer
- **Creates volume storage** for source data processing:
    - `sales` volume: Active processing location for pipeline consumption
    - `ops` volume: Staging location containing multiple data files
- **Verifies compute environment** compatibility with Lakeflow pipelines

This ensures that all schemas, tables, and objects are created in your designated catalog with proper isolation.

**Important:** You must have permission to create catalogs in your own non-Vocareum workspace. If you do not have the required permissions, this step will fail. Review the troubleshooting note below before continuing.


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">

  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting Setup - Missing Create Catalog Permissions
  </strong>
<details>
  <div style="color:#333;">

If you do not have permission to create a new catalog but already have one available, you can explicitly specify an existing catalog by using the `catalog_forced` argument in the `build_user_catalog_name` function.

This function is defined in the notebook: `./Includes/Classroom-Setup-dq`

  </div>
</details>
</div>





In [0]:
%run ./Includes/Classroom-Setup-dq

### A2. Verify Volume Path Configuration

Run the cell below to view the value of the `my_vol_path` variable and confirm that it references your **your-catalog.dq_1_bronze** path. This path will be used to dynamically reference your source volumes throughout this demonstration.

In [0]:
print(my_vol_path)

## B. Explore Source Data Characteristics

Before building the pipeline, examine the source data to understand its structure, quality characteristics, and schema evolution patterns. This analysis will inform our quality expectation design.

### B1. Examine Available Data Files

View all files available in your **ops** volume. Note that it contains multiple files that will be incrementally moved to the `sales` volume for processing, simulating real-world data arrival patterns.

In [0]:
# List files in volume
display(spark.sql(f"LIST '{my_vol_path}/ops'"))

View the current files available for processing in the **sales** volume location. The `sales` volume currently contains only one file; additional files will be added incrementally from the `ops` location to demonstrate schema evolution and quality issue detection.

In [0]:
# List files in volume
display(spark.sql(f"LIST '{my_vol_path}/sales'"))

### B2. Analyze First File - Clean Baseline Data

Explore the first file which contains clean, high-quality data that will establish our quality baseline and validate the pipeline framework.

In [0]:
%sql
SELECT * 
FROM read_files(
    my_vol_path || '/ops/sales_1.csv'
);

**Firt File Characteristics (sales_1.csv):**
- **157 clean records** from a single sales subsidiary with perfect data quality
- **16 columns**: Standard order fields including order_id, customer_id, qty, unit_price, discount_pct, total_amount, order_date
- **Quality Profile**: No missing business keys, valid discount percentages (0-100%), recent order dates
- **Purpose**: Establishes 100% quality baseline to validate expectation framework

### B3. Analyze Second File - Quality Issues and Schema Evolution

Explore the second file which contains intentional quality issues and demonstrates schema evolution with new business columns.

In [0]:
%sql
SELECT * 
FROM read_files(
    my_vol_path || '/ops/sales_2.csv'
);

**Second File Characteristics (sales_2.csv):**

**Schema Evolution (New Business Columns):**
- **19 columns** (16 original + 2 new business fields)
- **New Column**: `order_status` (confirmed, pending, shipped, delivered, cancelled)
- **New Column**: `shipping_cost` (numeric values for shipping fees)

**Intentional Quality Issues for Testing:**
- **Invalid discount percentages**: discount_pct = 120 (violates discount <= 100% rule)
- **Negative discounts**: discount_pct = -10.73 (violates discount >= 0 rule)
- **Historical dates**: order_date = 1930-12-31 (violates recent date range requirement)
- **Missing business keys**: NULL order_id or customer_id values
- **Invalid shipping costs**: shipping_cost > 100 or shipping_cost < 0

**Expected Quality Results:**
- Approximately 85% of records will pass all quality checks
- 15% will be quarantined with detailed failure reasons for remediation

**Data Analysis Checkpoint:**
- **sales_1.csv**: 157 rows (clean baseline for framework validation)
- **sales_2.csv**: 60 rows (quality issues + schema evolution demonstration)
- **Total Expected**: 217 records across both files with varying quality profiles

## C. Create the Spark Declarative Pipeline

Now that we understand the data characteristics and quality challenges, create the Spark Declarative Pipeline with comprehensive data quality expectations and quarantine handling.

### C1. Enable the Lakeflow Pipelines Editor

Complete the following steps to confirm or enable the **Lakeflow Pipelines Editor**:

1. In the top-right corner of the workspace, select your **account icon** ![Account Icon](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/account_icon.png) (*Your icon letter will differ*).  

2. Right-click **Settings** and choose **Open link in new tab**.  

3. In the left sidebar, select **Developer** under **User**.  

4. In the **Experimental features** section, locate **Lakeflow Pipelines Editor** and toggle it **on**.

### C2. Create a Apache Spark™ Declarative Pipeline

Complete the following steps to create your Spark Declarative Pipeline using the Lakeflow Pipelines Editor:

1. In the main navigation pane, right-click **Jobs & Pipelines** and select **Open link in New Tab**.  

2. In the new tab, select **Create → ETL Pipeline**.  

   **NOTE:** If prompted to **Try the new Lakeflow Pipelines Editor**, choose **Enable Lakeflow Pipelines Editor**. This appears only if you did not complete the previous step.  

3. Configure the pipeline settings:
   - **Pipeline Name**: `demo_dq_yourname`
   - **Default Catalog**: `YOUR_LABUSER_CATALOG`
   - **Default Schema**: `dq_1_bronze`  
   **NOTE:** Clear the selected schema using the cross icon to view all schemas.

4. **Rename pipeline components**:
   - Rename the **transformations** folder to `dq_pipeline`
   - Rename **my_transformation.py** file to `bronze_ingestion.sql`

5. Leave the **Lakeflow Pipelines Editor** page open for the next steps.

### C3. Configure Pipeline Parameters

1. Run the cell below to retrieve the key-value pairs needed to set your pipeline configuration parameters for the **source volume**.

In [0]:
config_parameters = [
    ('source', f'{my_vol_path}/sales')
]

for key, value in config_parameters:
    print(f"Key: {key}\nValue: {value}\n")

2. Copy the paths above and add each one as a configuration parameter in your **Spark Declarative Pipeline**.

    This will allow your pipeline to reference each volume through parameters.

   a. Select **Settings** in your pipeline tab  

   b. Under **Configuration**, select **Add configuration**

   c. For each **Key**, enter the key name shown above  

   d. For each **Value**, enter the corresponding volume path  

   e. Select **Save**

**NOTE:** For more details on configuration parameters, see the Databricks documentation: [Use parameters with Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/parameters)

## D. Create Bronze Layer - Raw Ingestion with Schema Evolution

The bronze layer implements schema evolution by using the `read_files` function with flexible column handling and schema hints for future column compatibility.

### D1. Create Bronze Table with Schema Evolution Support

Copy the code below into your `bronze_ingestion.sql` file. This implementation uses a STRING-based approach for maximum schema flexibility and includes schema hints for future columns.

<button onclick="copyBlock()">Copy to clipboard</button>
<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>

------------------------------------------
-- BRONZE LAYER: RAW DATA INGESTION
------------------------------------------
CREATE OR REFRESH STREAMING TABLE dq_1_bronze.sales_bronze_raw_demo
AS
SELECT 
  -- Core business fields (all as STRING for schema flexibility)
  CAST(subsidiary_id AS STRING) AS subsidiary_id,
  CAST(order_id AS STRING) AS order_id,
  CAST(order_timestamp AS STRING) AS order_timestamp,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(region AS STRING) AS region,
  CAST(country AS STRING) AS country,
  CAST(city AS STRING) AS city,
  CAST(channel AS STRING) AS channel,
  CAST(sku AS STRING) AS sku,
  CAST(category AS STRING) AS category,
  CAST(qty AS STRING) AS qty,
  CAST(unit_price AS STRING) AS unit_price,
  CAST(discount_pct AS STRING) AS discount_pct,
  CAST(coupon_code AS STRING) AS coupon_code,
  CAST(total_amount AS STRING) AS total_amount,
  CAST(order_date AS STRING) AS order_date,

  -- New columns for schema evolution (will be null initially)
  CAST(order_status AS STRING) AS order_status,
  CAST(shipping_cost AS STRING) AS shipping_cost,

  -- Rescued data column for parsing issues
  CAST(_rescued_data AS STRING) AS _rescued_data,

  -- Metadata columns for lineage tracking
  _metadata.file_name AS source_file,
  _metadata.file_modification_time AS file_mod_time
FROM STREAM read_files(
  '${source}',
  format => 'csv',
  schemaHints => 'order_status STRING, shipping_cost STRING'
);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Bronze Layer Design Patterns:**

**Schema Evolution Strategy:**
- **STRING storage**: All business columns stored as STRING for maximum flexibility and compatibility
- **Pre-defined schema**: Includes both original and two future columns for seamless evolution
- **Schema hints**: Declares expected new columns (`order_status`, `shipping_cost`) even when not yet present in data
- **Null handling**: New columns will be NULL for existing files, populated for future files

**Data Lineage and Protection:**
- **Auto Loader**: `read_files` function enables incremental processing with checkpoint management
- **Metadata capture**: Source file names and modification timestamps for complete data lineage
- **Rescued data**: `_rescued_data` column captures any parsing issues for investigation and recovery
- **Parameter reference**: `${source}` enables dynamic path configuration

**Technical Benefits:**
- No pipeline code changes required when new columns arrive
- Backward compatibility with existing data files
- Forward compatibility with evolving business requirements

For more information about schema hints functionality, see the [Databricks documentation on schema hints](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema#override-schema-inference-with-schema-hints).

### D2. Create Clean Bronze Table

Create a streaming intermediate Bronze table that performs safe type casting from the raw ingested data, without applying any quality checks. This step ensures clean type conversion before data validation.

- Creates a clean streaming Bronze table from the source data stream.
- Uses `TRY_CAST` to safely convert data types (dates, numbers) without failing on bad data.
- Passes through all columns without applying any data quality filters yet.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- BRONZE LAYER: INTERMEDIATE TRANSFORMATION TABLE
-- Purpose: Clean type casting from Bronze Raw (no quality checks yet)
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_1_bronze.sales_bronze_clean_demo
COMMENT "Intermediate table - type casting only, no quality checks"
AS
SELECT
  -- Direct pass-through and safe type casting
  subsidiary_id,
  order_id,
  TRY_CAST(order_timestamp AS TIMESTAMP) AS order_timestamp,
  TRY_CAST(order_date AS DATE) AS order_date,
  customer_id,
  region,
  country,
  city,
  channel,
  sku,
  category,
  TRY_CAST(qty AS INT) AS qty,
  TRY_CAST(unit_price AS DOUBLE) AS unit_price,
  TRY_CAST(discount_pct AS DOUBLE) AS discount_pct,
  TRY_CAST(total_amount AS DOUBLE) AS total_amount,
  coupon_code,
  order_status,
  TRY_CAST(shipping_cost AS DOUBLE) AS shipping_cost,
  source_file

FROM STREAM dq_1_bronze.sales_bronze_raw_demo;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### D3. Run and Validate the Bronze Layer

1. **Run the pipeline** and confirm that it completes successfully with no errors.

2. **Validate ingestion results** in the Lakeflow Pipelines editor:
   - Verify that **157 records** were ingested into the sales_bronze_raw_demo and table from the first file
   - Select the **sales_bronze_raw_demo** table and open the **Data** tab to preview the ingested sales data
   - Confirm that new columns (`order_status`, `shipping_cost`) exist but contain NULL values

**Troubleshooting:** If your pipeline fails, verify that:
- Volume paths are correctly configured in pipeline parameters
- The `source` configuration parameter matches your volume path
- Serverless compute is selected and running

**Expected Checkpoint Results:**

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/checkpoint_bronze.png" alt="Bronze Layer Results" width="1200">

### D4. Query and Analyze Bronze Layer

Run these queries to validate the bronze layer tables:

- Confirm that all records are ingested into the raw bronze table
- Check that `schema_hints` worked successfully

In [0]:
%sql
-- Verify record ingestion by source file (should show only sales_1.csv)
SELECT 
  source_file,
  COUNT(*) AS record_count,
  'Clean baseline data' AS data_profile
FROM dq_1_bronze.sales_bronze_raw_demo
GROUP BY source_file
ORDER BY source_file;

In [0]:
%sql
-- Verify schema evolution readiness
-- Total Column should be 21
DESCRIBE TABLE dq_1_bronze.sales_bronze_raw_demo;

## E. Create Silver Layer with Data Quality Expectations and Quarantine Logic

- In this layer, we will first type-cast columns to their appropriate data formats.
- Next, we will apply data quality expectations.
- Finally, based on these expectations, we will separate the data into two tables: one for clean records and one for invalid records, using inverse logic.

### E1. Understanding Data Quality Expectations Framework

Before creating tables with expectations, understand the expectation framework and why each validation is critical for enterprise data quality.

**Expectation Syntax:**
```sql
CONSTRAINT <constraint_name> 
  EXPECT (<condition>) 
  ON VIOLATION <action>
```

**Available Violation Actions:**
- **WARN** (default): Invalid records included but violation logged for monitoring
- **DROP ROW**: Invalid records excluded from target table 
- **FAIL UPDATE**: Pipeline stops and requires manual intervention

**Our Strategy:** We use **WARN** (default) for all expectations to ensure the pipeline continues running and all records are retained for quarantine analysis.

**Our 6 Comprehensive Quality Expectations:**

**Category 1: Business Key Validation (Critical for Data Integrity)**
1. **`check_subsidiary_id`**: Every order must have a subsidiary identifier for proper attribution
2. **`check_customer_id`**: Must identify which customer placed the order for analytics and compliance
3. **`check_sku`**: Must identify what product was ordered for inventory and revenue tracking

**Category 2: Range-Based Validations (Business Rule Enforcement)**

4. **`valid_discount_range`**: Discount must be 0%-100% (catches negative discounts and impossible values)
5. **`valid_date_range`**: Order dates within last 4 years  (catches archived data and future dates)

**NOTE:** Since our data is static, we are checking the constraint **`valid_date_range`** with respect to **01-01-2026**.

**Category 3: Schema Evolution Validation (New Column Quality)**

6. **`valid_shipping_cost`**: Shipping cost must be between $0 and $100 (validates new business column)

**Note on Schema Evolution:** The `valid_shipping_cost` expectation applies to a column not present in initial data but expected in future files. Once the column appears, the expectation automatically enforces validation. For details, see [Schema Evolution Pattern](https://docs.databricks.com/aws/en/ldp/expectation-patterns?language=SQL#schema-evolution-pattern).

### E2. Understanding Quarantine Records Pattern

**Quarantine records** in Spark Declarative Pipelines are records that violate data quality expectations and are routed to a separate quarantine table for analysis and remediation. This enterprise pattern ensures:

**Benefits:**
- **Zero data loss**: Invalid data is isolated for review, not dropped or causing pipeline failures
- **Detailed tracking**: Metadata captures which specific expectations failed for each record
- **Flexible workflows**: Enables remediation processes, reprocessing, and quality improvement
- **Audit compliance**: Complete record of data quality issues for regulatory requirements

**Implementation Pattern:**
- **Primary table**: Contains expectations with WARN action (logs violations, retains records)
- **Quarantine logic**: Uses inverse logic (NOT condition) to identify failed records
- **Parallel processing**: Valid and invalid records flow to separate tables simultaneously

> **Learn More:**  
> 💡 For comprehensive information on quarantine frameworks and best practices, see the [Databricks documentation on expectation patterns and quarantining invalid records](https://docs.databricks.com/aws/en/ldp/expectation-patterns?language=Python%20Module#quarantine-invalid-records).

### E3. Create a New SQL File in your Pipeline

1. Click the kebab menu next to your `dq_pipeline` folder and select **Create file**.
2. Select the language as **SQL**.
3. Name the file `silver_transformation.sql`.

### E4. Create Quarantine Table with Quality Expectations


Add the following code to your `silver_transformations.sql` file. This creates the main quarantine table with all 6 data quality expectations defined.

<button onclick="copyBlock()">Copy to clipboard</button>
<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- SILVER LAYER - STEP 1: QUARANTINE TABLE WITH EXPECTATIONS
-- Purpose: Define table structure and 6 data quality expectations
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_2_silver.sales_silver_dq_demo
(
  -- Business columns with proper data types
  subsidiary_id STRING,
  order_id STRING,
  order_timestamp TIMESTAMP,
  order_date DATE,
  customer_id STRING,
  region STRING,
  country STRING,
  city STRING,
  channel STRING,
  sku STRING,
  category STRING,
  qty INT,
  unit_price DOUBLE,
  discount_pct DOUBLE,
  total_amount DOUBLE,
  coupon_code STRING,
  order_status STRING,
  shipping_cost DOUBLE,
  source_file STRING,

  -- Quality tracking columns for quarantine analysis
  is_quarantined BOOLEAN,
  quarantine_reason STRING,

  --========================================================================
  -- 6 DATA QUALITY EXPECTATIONS (WARN ACTION FOR QUARANTINE PATTERN)
  --========================================================================

  CONSTRAINT check_subsidiary_id 
    EXPECT (subsidiary_id IS NOT NULL),

  CONSTRAINT check_customer_id 
    EXPECT (customer_id IS NOT NULL),

  CONSTRAINT check_sku 
    EXPECT (sku IS NOT NULL),

  CONSTRAINT valid_discount_range
    EXPECT (discount_pct IS NULL OR (discount_pct >= 0 AND discount_pct <= 100)),

  -- Since the data is static, we check that order_date falls between 01-01-2026 and 4 years prior, instead of using the current date.
  CONSTRAINT valid_date_range 
    EXPECT (order_date IS NULL OR 
        (order_date >= DATE_SUB(DATE '2026-01-01', 1460) AND 
          order_date <= DATE '2026-01-01')),

  CONSTRAINT valid_shipping_cost EXPECT (
    CASE WHEN shipping_cost IS NOT NULL THEN shipping_cost > 0 AND shipping_cost < 100 ELSE TRUE END
  )
)
COMMENT "Quarantine table with 6 expectations - supports inverse logic pattern"
TBLPROPERTIES (
  'quality.layer' = 'silver_quarantine',
  'quality.pattern' = 'inverse_logic'
)
PARTITIONED BY (is_quarantined);
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Expectation Framework:**

**Expectations Summary:**

| # | Expectation Name | Validates | Range/Rule | Violation Example |
|---|-----------------|-----------|------------|---------------------|
| 1 | check_subsidiary_id | Business Key | NOT NULL | Missing subsidiary IDs |
| 2 | check_customer_id | Business Key | NOT NULL | Missing customer IDs |
| 3 | check_sku | Business Key | NOT NULL | Missing SKUs |
| 4 | valid_discount_range | Discount | 0%-100% | discount = -10.73% or 120% |
| 5 | valid_date_range | Date | Last 4 years | date = 1930-12-31 |
| 6 | valid_shipping_cost | Shipping | $0-$100 | shipping_cost = -5 or 150 |

**Quality Tracking Design:**
- **is_quarantined**: Boolean flag indicating whether record failed any expectation
- **quarantine_reason**: Detailed string listing all failed validations for remediation
- **Partitioning**: Separates quarantined and valid data for performance optimization

**Table Properties:**
- Metadata tags identify this as a quarantine pattern implementation
- Enables monitoring and governance tools to recognize the quality framework

### E5. Create Inverse Logic Flow for Quarantine Population

Add the inverse logic flow to populate the quarantine table with quality tracking information. This flow implements the official Databricks quarantine pattern by using inverse logic to identify and flag records that fail any quality expectations.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- SILVER LAYER - STEP 2: FLOW TO APPLY INVERSE LOGIC
-- Purpose: Calculate is_quarantined flag and populate quarantine table
--========================================================================

CREATE FLOW apply_inverse_logic_flow
AS
INSERT INTO dq_2_silver.sales_silver_dq_demo BY NAME
SELECT
  -- Pass through all business columns
  subsidiary_id,
  order_id,
  order_timestamp,
  order_date,
  customer_id,
  region,
  country,
  city,
  channel,
  sku,
  category,
  qty,
  unit_price,
  discount_pct,
  total_amount,
  coupon_code,
  order_status,
  shipping_cost,
  source_file,

  --========================================================================
  -- INVERSE LOGIC: Mark as quarantined if ANY expectation fails
  --========================================================================
  NOT (
    (subsidiary_id IS NOT NULL) AND
    (customer_id IS NOT NULL) AND
    (sku IS NOT NULL) AND
    (discount_pct IS NULL OR (discount_pct >= 0 AND discount_pct <= 100)) AND
    (order_date IS NULL OR 
     (order_date >= DATE_SUB(DATE '2026-01-01', 1460) AND 
      order_date <= DATE '2026-01-01')) AND
    (shipping_cost IS NULL OR (shipping_cost > 0 AND shipping_cost < 100))
  ) AS is_quarantined,

  --========================================================================
  -- BUILD DETAILED QUARANTINE REASON STRING
  --========================================================================
  CONCAT_WS('; ',
    CASE WHEN subsidiary_id IS NULL 
         THEN 'Missing subsidiary_id' END,
    CASE WHEN customer_id IS NULL 
         THEN 'Missing customer_id' END,
    CASE WHEN sku IS NULL 
         THEN 'Missing sku' END,
    CASE WHEN discount_pct IS NOT NULL AND (discount_pct < 0 OR discount_pct > 100) 
         THEN 'Invalid discount_pct (must be 0-100)' END,
    CASE WHEN order_date IS NOT NULL AND 
              (order_date < DATE_SUB(DATE '2026-01-01', 1460) OR order_date > DATE '2026-01-01') 
         THEN 'Invalid order_date (outside 4-year range)' END,
    CASE WHEN shipping_cost IS NOT NULL AND (shipping_cost <= 0 OR shipping_cost >= 100)
         THEN 'Invalid shipping_cost (must be 0-100)' END
  ) AS quarantine_reason

FROM STREAM dq_1_bronze.sales_bronze_clean_demo;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Inverse Logic Implementation:**

**Quarantine Flag Logic:**
- **NOT()** wrapper around all expectations combined with AND logic
- If ANY expectation fails, the record is marked `is_quarantined = TRUE`
- If ALL expectations pass, the record is marked `is_quarantined = FALSE`

**Detailed Failure Tracking:**
- **CONCAT_WS()** builds semicolon-separated failure reasons
- Each CASE statement checks for specific violation types
- Provides actionable information for data remediation teams
- Empty string for records that pass all validations

**Streaming Flow Benefits:**
- Processes records continuously as they arrive
- Maintains low latency for real-time quality monitoring
- Enables immediate quarantine identification for operational workflows

### E6. Create Separate Tables for Valid and Quarantined Records

Create separate tables for valid and quarantined records to enable zero data loss and parallel processing workflows. Add this code to your `silver_transformations.sql` file.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- SILVER LAYER - STEP 3: SEPARATE VALID AND INVALID DATA PATHS
--========================================================================

--========================================================================
-- VALID RECORDS PATH - Clean data for analytics
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_2_silver.sales_silver_valid_demo
COMMENT "Clean records passing all 6 quality checks - ready for analytics"
AS
SELECT * EXCEPT (is_quarantined, quarantine_reason)
FROM STREAM dq_2_silver.sales_silver_dq_demo
WHERE is_quarantined = FALSE;

--========================================================================
-- QUARANTINED RECORDS PATH - Invalid data for remediation
--========================================================================

CREATE OR REFRESH STREAMING TABLE dq_2_silver.sales_silver_quarantined_demo
COMMENT "Invalid records with quality violations - requires remediation"
AS
SELECT *
FROM STREAM dq_2_silver.sales_silver_dq_demo
WHERE is_quarantined = TRUE;

</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review the Official Quarantine Pattern Implementation:**

**Zero Data Loss Architecture:**
- **Source table**: Contains expectations with WARN action (logs violations, retains all records)
- **Valid path**: Records where `is_quarantined = FALSE` (passed all quality checks)
- **Quarantine path**: Records where `is_quarantined = TRUE` (failed one or more checks)
- **Mathematical guarantee**: Total Records = Valid Records + Quarantined Records

**Parallel Processing Benefits:**
- **Analytics workflow**: Consumes only validated, clean data from `sales_silver_valid_demo`
- **Remediation workflow**: Processes quarantined data with detailed failure information
- **Monitoring workflow**: Tracks quality metrics across both streams
- **Performance optimization**: Separate tables enable independent scaling and optimization

## F. Create Gold Layer - Production-Ready Analytics Data

The gold layer contains pre-aggregated analytics from validated silver data, providing fast query performance for business intelligence and reporting applications.

### F1. Create Gold Layer Analytics Table

1. In your **dq_pipeline** folder, select the kebab menu and select **Create File**
2. Select the language as **SQL**
3. Name the file `gold_analytics.sql`
4. Copy and paste the code below

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
--========================================================================
-- GOLD LAYER: MATERIALIZED VIEW FOR BUSINESS ANALYTICS
--========================================================================

CREATE OR REFRESH MATERIALIZED VIEW dq_3_gold.sales_analytics_demo
COMMENT "Business metrics aggregated from validated sales data"
AS
SELECT
  region,
  country,
  category,
  COUNT(DISTINCT order_id) AS total_orders,
  COUNT(DISTINCT customer_id) AS unique_customers,
  SUM(qty) AS total_quantity_sold,
  ROUND(SUM(total_amount), 2) AS total_revenue,
  ROUND(AVG(total_amount), 2) AS avg_order_value,
  ROUND(AVG(discount_pct), 2) AS avg_discount_pct,
  MIN(order_date) AS earliest_order_date,
  MAX(order_date) AS latest_order_date,
  CURRENT_TIMESTAMP() AS last_refreshed_at
FROM dq_2_silver.sales_silver_valid_demo
GROUP BY region, country, category;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

**Review Gold Layer Design for Analytics:**

**Why Materialized Views for Gold Layer?**
- **Pre-computed aggregations**: Eliminates expensive GROUP BY operations at query time
- **Automatic refresh**: Updates when underlying silver data changes
- **Delta Lake storage**: Provides ACID guarantees and time travel capabilities
- **BI tool optimization**: Perfect for dashboards requiring fast response times

**Business Metrics Provided:**
- **Revenue analytics**: Total revenue, average order value by region and category
- **Customer insights**: Unique customer counts and purchasing patterns
- **Product performance**: Quantity sold and discount effectiveness by category
- **Temporal analysis**: Order date ranges for trend analysis
- **Data freshness**: Last refresh timestamp for monitoring

**Analytics Workflow Support:**
- **BI dashboards**: Query pre-aggregated gold views for fast performance
- **Data science**: Access detailed, validated silver tables for modeling
- **Operations teams**: Monitor quarantine tables for data quality remediation
- **Executive reporting**: Use gold metrics for strategic decision making

## G. Run and Analyze Pipeline Results - First Run (Clean Baseline)

Execute the complete pipeline and analyze the results to validate that the quality framework correctly processes clean baseline data.

### G1. Execute the Complete Pipeline

1. **Run the pipeline** and confirm that it completes successfully with no errors
2. **Observe the pipeline graph** to see data flowing through all layers: Bronze → Silver → Gold
3. **Monitor execution time** and resource utilization in the pipeline UI

**Troubleshooting:** If your pipeline fails, verify that:
- All SQL files are properly created in the `dq_pipeline` folder
- Configuration parameters are correctly set
- Serverless compute is selected and running
- Volume paths are accessible and contain data files

**Expected Pipeline Graph:**

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/checkpoint_run_1.png" alt="Complete Pipeline Run 1" width="1200">

### G2. Validate First Run Results - Clean Baseline

Review the pipeline execution to confirm that all data quality checks passed and the pipeline established a perfect baseline. This run demonstrates that the pipeline framework correctly ingests, transforms, and validates clean data.

**Expected Results for Run 1 (Clean Baseline):**

| Table | Expected Records | Quality Status | Notes |
|-------|------------------|----------------|-------|
| Bronze (sales_bronze_raw_demo) | 157 | All ingested | Raw data from sales_1.csv |
| Silver (sales_bronze_clean_demo) | 157 | Type-cast success | All conversions successful |
| Silver (sales_silver_dq_demo) | 157 | All tracked | All records with is_quarantined = FALSE |
| Silver (sales_silver_valid_demo) | 157 | 100% quality | Perfect baseline established |
| Silver (sales_silver_quarantined_demo) | 0 | No violations | Quality framework validated |
| Gold (sales_analytics_demo) | ~55 | Aggregated | Business metrics by region/category |

**Validation Steps in Pipeline UI:**
1. Select the **sales_silver_dq_demo** table and open the **Expectations** tab to verify "6 constraints tracked"
2. Open the **Data** tab to confirm all records have `is_quarantined = FALSE`
3. Check the **sales_silver_quarantined_demo** table to ensure it contains 0 records
4. Verify the **sales_analytics_demo** gold table contains aggregated business metrics

### G3. Query and Analyze Results

Execute the following queries to verify silver and gold layers result:
- Records in **sales_silver_quarantined_demo** should be zero
- Quality Score should be **100%**
- Querying on **sales_analytics_demo** MV for insight

In [0]:
%sql
-- Verify quarantine is empty for Run 1 (should return 0 - perfect quality baseline)
SELECT 
  COUNT(*) AS quarantined_count,
  'Expected: 0 records for clean baseline' AS validation_note
FROM dq_2_silver.sales_silver_quarantined_demo;

In [0]:
%sql
-- Verify all records passed quality checks
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN is_quarantined = FALSE THEN 1 ELSE 0 END) AS valid_records,
  SUM(CASE WHEN is_quarantined = TRUE THEN 1 ELSE 0 END) AS quarantined_records,
  ROUND(SUM(CASE WHEN is_quarantined = FALSE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS quality_score_pct
FROM dq_2_silver.sales_silver_dq_demo;

In [0]:
%sql
--Querying on sales_analytics materailized view
SELECT * FROM dq_3_gold.sales_analytics_demo

**Run 1 Validation Checkpoint - Expected Results:**
- **Bronze Layer**: 157 records from sales_1.csv successfully ingested
- **Silver Transformation**: 157 records type-cast successfully with no conversion errors
- **Quality Validation**: 157 records pass all 6 expectations (100% quality score)
- **Quarantine Status**: 0 records quarantined (perfect baseline established)
- **Gold Analytics**: Business metrics aggregated should have 55 validated records

## H. Prepare and Execute Second Run - Quality Issues and Schema Evolution

Introduce the second data file containing quality issues and new schema columns to demonstrate the pipeline's resilience and monitoring capabilities.

### H1. Copy Additional Files for Processing

Copy the second file from the operations location to the sales processing location to trigger incremental processing with quality issues and schema evolution.

In [0]:
ops_path = f'/Volumes/{my_catalog}/dq_1_bronze/ops'
sales_path = f'/Volumes/{my_catalog}/dq_1_bronze/sales'

# Copy files from ops location to user's source volume (now including sales_2.csv)
copy_files(copy_from=ops_path, copy_to=sales_path, n=2)

### H2. Verify File Availability for Processing

Confirm that `sales_2.csv` has been successfully moved to the processing source location and is ready for pipeline consumption.

In [0]:
spark.sql(f"LIST '{sales_path}'").display()

### H3. Execute Pipeline Run 2 - Quality Issues and Schema Evolution

1. **Run the pipeline again** and confirm that it completes successfully despite quality issues
2. **Observe incremental processing** as the pipeline detects and processes the new file
3. **Monitor quality metrics** in the pipeline UI to see expectation violations being tracked

**Expected Behavior:**
- Pipeline continues running despite quality violations (WARN action)
- New columns are automatically incorporated (schema evolution)
- Invalid records are systematically quarantined with detailed failure reasons
- Valid records continue to flow to analytics tables

**Expected Pipeline Results:**

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/dq/checkpoint_run_2.png" alt="Pipeline Run 2 with Quality Issues" width="1200">

### H4. Analyze Run 2 Results - Quality Issues Detected

Review the pipeline execution to see how quality issues and schema evolution were successfully handled by the quarantine framework.

**Expected Results for Run 2 (Incremental with Issues):**

| Table | New Records | Total Records | Quality Notes |
|-------|-------------|---------------|---------------|
| Bronze (sales_bronze_raw_demo) | +60 | 217 | Schema evolved! (2 new columns populated) |
| Bronze (sales_bronze_clean_demo) | +60 | 217 | Type casting successful for all records |
| Silver (sales_silver_dq_demo) | +60 | 217 | All records tracked with quality flags |
| Silver (sales_silver_valid_demo) | +51 | 208 | 9 records failed expectations |
| Silver (sales_silver_quarantined_demo) | +9 | 9 | Invalid records captured with failure details |
| Gold (sales_analytics_demo) | Updated | ~60 | Metrics updated with new valid data |

**Key Validation Points:**
1. **Zero Data Loss**: Bronze (217) = Valid (208) + Quarantined (9)
2. **Schema Evolution**: New columns populated automatically
3. **Quality Detection**: Specific violations identified and quarantined
4. **Pipeline Resilience**: Continues processing despite quality issues

### H5. Comprehensive Quality Analysis

Execute detailed analysis queries to understand the quality patterns and validate the quarantine framework effectiveness.

In [0]:
%sql
-- Overall quality metrics and zero data loss verification
WITH quality_stats AS (
  SELECT 
    (SELECT COUNT(*) FROM dq_1_bronze.sales_bronze_raw_demo) AS total_ingested,
    (SELECT COUNT(*) FROM dq_2_silver.sales_silver_valid_demo) AS total_valid,
    (SELECT COUNT(*) FROM dq_2_silver.sales_silver_quarantined_demo) AS total_quarantined
)
SELECT 
  total_ingested,
  total_valid,
  total_quarantined,
  ROUND(total_valid * 100.0 / total_ingested, 2) AS quality_score_pct,
  ROUND(total_quarantined * 100.0 / total_ingested, 2) AS failure_rate_pct,
  CASE WHEN total_ingested = total_valid + total_quarantined 
       THEN '✅ ZERO DATA LOSS VERIFIED' 
       ELSE '❌ DATA LOSS DETECTED' 
  END AS data_loss_check
FROM quality_stats;

In [0]:
%sql
-- Analyze specific quarantined records with detailed failure reasons
SELECT 
  order_id,
  customer_id,
  discount_pct,
  order_date,
  shipping_cost,
  source_file,
  quarantine_reason
FROM dq_2_silver.sales_silver_quarantined_demo
ORDER BY quarantine_reason, order_id;

**Quality Analysis Insights:**
- **Zero Data Loss Achieved**: Mathematical verification that no records were lost
- **High Quality Score**: Typically 94-96% overall quality across both files
- **Detailed Failure Tracking**: Each quarantined record includes specific violation reasons
- **Actionable Intelligence**: Operations teams can prioritize remediation based on failure types
- **Range Validation Effectiveness**: Successfully caught discount > 100%, negative values, and historical dates
- **Business Key Protection**: Identified and quarantined records with missing critical identifiers

### H6. Validate Schema Evolution Success

Verify that schema evolution worked seamlessly without any pipeline code changes. Note that records from the first file will have null values for the new columns(**shipping_cost** and **order_status**), while records from the second file will have populated values.

In [0]:
%sql
SELECT *
FROM dq_2_silver.sales_silver_valid_demo

**Schema Evolution Success Validation:**
- **Seamless Evolution**: Schema grew from 16 to 18 core business columns without any pipeline code changes
- **Backward Compatibility**: File 1 records maintain NULL values for new columns
- **Forward Compatibility**: File 2 records populate new columns with business values
- **No Pipeline Disruption**: Schema hints enabled graceful column addition
- **Business Value Addition**: New columns provide order status tracking and shipping cost analysis
- **Quality Framework Extension**: New columns automatically included in quality validations

### H7. Quality Performance by Source File

Analyze quality performance differences between the clean baseline file and the file with intentional quality issues.

In [0]:
%sql
-- Quality score analysis by source file
SELECT 
  COALESCE(b.source_file, q.source_file) AS source_file,
  COUNT(DISTINCT COALESCE(b.order_id, q.order_id)) AS total_records,
  COUNT(DISTINCT s.order_id) AS valid_records,
  COUNT(DISTINCT q.order_id) AS quarantined_records,
  ROUND(COUNT(DISTINCT s.order_id) * 100.0 / 
        NULLIF(COUNT(DISTINCT COALESCE(b.order_id, q.order_id)), 0), 2) AS quality_score_pct,
  CASE 
    WHEN COALESCE(b.source_file, q.source_file) LIKE '%sales_1%' THEN 'Clean baseline data'
    WHEN COALESCE(b.source_file, q.source_file) LIKE '%sales_2%' THEN 'Quality issues + schema evolution'
    ELSE 'Unknown file pattern'
  END AS file_profile
FROM dq_1_bronze.sales_bronze_raw_demo b
FULL OUTER JOIN dq_2_silver.sales_silver_valid_demo s 
  ON b.order_id = s.order_id
FULL OUTER JOIN dq_2_silver.sales_silver_quarantined_demo q 
  ON COALESCE(b.order_id, s.order_id) = q.order_id
GROUP BY COALESCE(b.source_file, q.source_file)
ORDER BY source_file;

**File-Level Quality Performance Insights:**
- **File 1 (Baseline)**: 100% quality score validates framework correctness
- **File 2 (Issues)**: 85% quality score demonstrates issue detection capability
- **Overall Performance**: 95.85% combined quality score shows enterprise-grade data quality
- **Framework Validation**: Clean data passes completely, problematic data is systematically identified
- **Production Readiness**: Quality framework successfully handles both perfect and imperfect data scenarios

## I. Summary and Key Takeaways

**Data Quality Management:**
- Successfully implemented 6 comprehensive data quality expectations covering business keys, numeric ranges, and temporal validations
- Achieved 100% quality score on clean baseline data (Run 1: 157/157 records)
- Detected and quarantined 9 quality violations in problematic data (Run 2: 51/60 records passed)
- Overall pipeline quality score: 95.85% (208/217 total records processed)

**Schema Evolution:**
- Seamlessly handled schema evolution from 16 to 18 core business columns from source without pipeline code changes
- Bronze layer STRING strategy enabled backward compatibility and forward flexibility
- New business columns (**order_status, shipping_cost**) automatically integrated

**Zero Data Loss Quarantine:**
- Implemented official Databricks inverse logic pattern for quarantine processing
- Achieved mathematical zero data loss: Bronze Records = Valid Sales Records + Quarantine Sales Records
- Provided detailed failure analysis and remediation insights for each quarantined record

### Production Readiness

This demonstration showcased enterprise-grade patterns that provide:
- **Robust data quality controls** that catch anomalies before they reach analytics
- **Flexible schema handling** that adapts to evolving business requirements
- **Complete data lineage** with zero loss and full audit capabilities
- **Actionable quality insights** for continuous data improvement

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>